<div dir="rtl" style="text-align:right">
<h1>Gradient جهت را نشان می‌دهد؛ اندازهٔ گام چه می‌کند؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چگونه جهت مشتق را از اندازهٔ گام جدا کنیم؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-02/chapter-05/11-derivative.html"><bdi dir="ltr">11-derivative</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-02/chapter-05/12-chain.html"><bdi dir="ltr">12-chain</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-02/chapter-05/12-sgd.html"><bdi dir="ltr">12-sgd</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html"><bdi dir="ltr">17-autograd</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، Kernel را Restart و سپس Run All کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">برای <code dir="ltr" style="unicode-bidi:isolate">L(w)=(2w−5)²</code> در w=1، پیش از اجرا علامت و مقدار مشتق را حساب کنید. backward فقط مشتق می‌سازد؛ خودش وزن را تغییر نمی‌دهد. در این دفتر، به‌روزرسانی Gradient descent را دستی می‌نویسیم.</p>
</div>

In [ ]:
w = torch.tensor(1., requires_grad=True)
loss = (2*w-5)**2
loss.backward()
print("Loss:", loss.item(), "Gradient:", w.grad.item(), "Weight:", w.item())
assert w.grad.item() == -12 and w.item() == 1
epsilon = 1e-4
f = lambda value: (2*value-5)**2
numerical = (f(1+epsilon)-f(1-epsilon))/(2*epsilon)
print("Finite difference:", numerical)
assert abs(numerical-w.grad.item()) < 1e-6
for lr in [0.1,0.5]:
    new_w = w.detach()-lr*w.grad
    print("lr:", lr, "new w:", new_w.item(), "new Loss:", f(new_w).item())


<div dir="rtl" style="text-align:right">
<h2>مسیر حرکت را ببینید</h2><p style="text-align:right">در این تابعِ مشخص، نرخ ۰٫۱ خطا را کم می‌کند؛ نرخ ۰٫۵ باعث دورشدن می‌شود. این عددها نسخهٔ همیشگی انتخاب Learning Rate نیستند. کدام نرخ نوسان می‌سازد؟ پیش‌بینی و سپس اجرا کنید.</p>
</div>

In [ ]:
def descent(lr, steps=6):
    value = 1.0
    path = [value]
    for _ in range(steps):
        value -= lr * 4*(2*value-5)
        path.append(value)
    return path

fig, axes = plt.subplots(1,2,figsize=(9,3))
grid = torch.linspace(0,5,200)
axes[0].plot(grid, (2*grid-5)**2)
good = descent(0.1)
axes[0].plot(good, [f(v) for v in good], "o-")
axes[0].set(xlabel="w", ylabel="Loss", title="Steps on the Loss surface")
for lr in [0.1,0.5]:
    values = descent(lr)
    axes[1].plot(range(len(values)), [f(v) for v in values], "o-", label=str(lr))
axes[1].set(xlabel="Step", ylabel="Loss", yscale="log")
axes[1].legend(title="Learning rate")
plt.tight_layout()
plt.show()


<div dir="rtl" style="text-align:right">
<h2>انباشته‌شدن مشتق و قطع مسیر</h2><p style="text-align:right">دو بار با Graph تازه backward می‌زنیم، بی‌آنکه مشتق را پاک کنیم. انتظار چه عددی دارید؟ سپس مشتق را پاک کنید. خطای no_grad در پایان عمدی است؛ کد آن را می‌گیرد تا اجرای دفتر متوقف نشود.</p>
</div>

In [ ]:
w = torch.tensor(1., requires_grad=True)
for _ in range(2):
    ((2*w-5)**2).backward()
assert w.grad.item() == -24
print("Accumulated:", w.grad.item())
w.grad = None
((2*w-5)**2).backward()
assert w.grad.item() == -12
with torch.no_grad():
    detached_loss = (2*w-5)**2
try:
    detached_loss.backward()
except RuntimeError as error:
    print("Expected missing Graph:", error)
else:
    raise AssertionError("Expected backward to fail")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> برای <code dir="ltr" style="unicode-bidi:isolate">(2w+3w)²</code> در w=1 مشتق را دستی و با Autograd حساب کنید. انتظار ۵۰ داریم، چون هر دو مسیر به w می‌رسند. توضیح دهید چرا پاک‌کردن Gradient با تغییر وزن یک کار نیست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl">
<h2>تمرین تکمیلی: مشتق میانگین خطاهای یک Batch</h2>
<p>چند مسیر مشتق را جمع کنید و اثر میانگین‌گیری را از Learning Rate جدا نگه دارید. پیش‌نیاز: Autograd و پاک‌کردن Gradient را در همین دفتر دیده‌اید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع TODO را خودتان بنویسید؛ INCOMPLETE یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>برای دو نمونه، Gradientِ میانگین Lossها با جمع Gradientها چه نسبتی دارد؟ تکرار کل Batch باید میانگین Gradient را عوض کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
features = torch.tensor([1.,3.])
targets = torch.tensor([2.,4.])
print('features/targets:',features,targets)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع batch_gradient(Weight,features,targets) برای میانگین (Weight*features-targets)^2 یک Gradient Scalar با Autograd حساب و float برگرداند. Weight ورودی عدد Python است؛ وزن را به‌روزرسانی نکنید.</p>
</div>

In [ ]:
def batch_gradient(weight, features, targets):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = batch_gradient(1.,features,targets)
    if result is None: return False
    assert math.isclose(result,-4.)
    assert math.isclose(batch_gradient(1.,features.repeat(3),targets.repeat(3)),-4.)
    a,b = torch.tensor([2.,-1.,3.]),torch.tensor([1.,2.,0.])
    expected = (2*(0.5*a-b)*a).mean().item()
    assert math.isclose(batch_gradient(0.5,a,b),expected,rel_tol=1e-6)
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Learning Rate را تغییر دهید؛ همان Gradient اولیه را نگه دارید. این مقایسه یک گام است و دربارهٔ کل مسیر آموزش ادعایی ندارد.</p>
</div>

In [ ]:
gradient = (2*(features-targets)*features).mean().item()
for learning_rate in (0.,0.1,0.5):
    updated = 1.-learning_rate*gradient
    print('learning rate, weight, next loss:',learning_rate,updated,((updated*features-targets)**2).mean().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>کد خراب برای میانگین Loss، Gradient جمع را به کار می‌برد. تابع mean_sample_gradients(gradients) را اصلاح کنید؛ فهرست ورودی غیرخالی است.</p>
</div>

In [ ]:
sample_gradients = [-2.,-6.]
print('wrong sum:',sum(sample_gradients),'sample count:',len(sample_gradients))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def mean_sample_gradients(gradients):
    # TODO
    return None

In [ ]:
def test_repair():
    result = mean_sample_gradients(sample_gradients)
    if result is None: return False
    assert result == -4.
    assert mean_sample_gradients([2.,-2.,6.]) == 2.
    assert mean_sample_gradients([5.]) == 5.
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>Loss پیش‌فرض MiniGPT میانگین موقعیت‌های Batch است. تغییر ناخواستهٔ mean به sum اندازهٔ Gradient را به تعداد هدف‌ها وابسته می‌کند، حتی اگر Learning Rate ثابت بماند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا مقایسهٔ دو Learning Rate بدون ثابت‌نگه‌داشتن قرارداد کاهش Loss می‌تواند گمراه‌کننده باشد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-04_gradients_steps.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>